In [1]:
using DelimitedFiles

file_path = "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/DatasetParams_2tonyr.txt"

# 会自动处理空格和制表符
data1 = readdlm(
    file_path;
)
# 提取行、列键
row_keys = [data1[i, 1] for i in 2:size(data1, 1)]
col_keys = [data1[1, j] for j in 2:size(data1, 2)]
dict_2d = Dict()
# 为每个行键创建列字典
for i in 2:size(data1, 1)
    row_key = data1[i, 1]
    row_dict = Dict()  
    for j in 2:size(data1, 2)
        col_key = data1[1, j]
        row_dict[col_key] = data1[i, j]
    end
    
    dict_2d[row_key] = row_dict
end
println("dict_2d[3801][\"E_cut\"] = ", dict_2d[3801]["E_cut"])

# using DataFrames, CSV

# # 读取文件到DataFrame
# df = CSV.read(
#     "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/DatasetParams_2tonyr.txt",
#     DataFrame, 
#     delim = "\t",
#     ignorerepeated=true,
#     comment="#",
#     # header=["DS", "E_cut", "err_cut", "E_mc", "err_mc", 
#           # "scaling", "err_scaling", "deltaT", "BI", "err_BI"],
#               # types=Dict(:DS=>Int64, :deltaT=>Float64)
# )
DIR = "/Users/zhaokangkang/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia"
SOURCE = "$(DIR)/src"
include("$(SOURCE)/readHist.jl")
include("$(SOURCE)/PhysicalConstants.jl")
include("$(SOURCE)/Lineshape.jl")
include("$(SOURCE)/Dataset.jl")
include("$(SOURCE)/HistogramAsUvDistribution.jl")
include("$(SOURCE)/Util.jl")

using .LineshapeModule
using .DatasetModule
using .PhysicalConstants
using .readHistModule
using .UtilsModule

using Distributions, IntervalSets
using OrderedCollections


ds_num_list= [3819, 3820]
Emin = 2465.0
Emax = 2575.0

path_peakshape = "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/lineshape/calibration/"
path_baselinesigma = "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/baseline_sigmas/"
path_superreduced = "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/super_reduced/background/unblinded/"
path_exposure = "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/Exposures/"
file_eff = "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/CombinedEfficiency/CombinedEfficiencies.root"
file_lsscaling = "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/lineshape_scaling_output/CombinedReso_model2_pol2bias.root"


datasets = Vector{DatasetModule.Dataset}()

for ds_num in ds_num_list
    LS1 = LineshapeGauss3(
        joinpath(path_peakshape, "PeakShape_ds$(ds_num).root"), 
        joinpath(path_baselinesigma, "bkg/ds$(ds_num)_baseline.root"),
        joinpath(path_baselinesigma, "calib/ds$(ds_num)_baseline.root"),
        Int(ds_num)
    )
    # set_poly_scaling(LS1, kQvalue, "/Users/zhaokangkang/gssiwork/julia-dev/code-test/test_json/lineshape_scaling_ds3021.json",2)
    # set_poly_scaling(LS1, kSigma, "/Users/zhaokangkang/gssiwork/julia-dev/code-test/test_json/lineshape_scaling_ds3021.json",2)
    SetHistoScaling(LS1, kQvalue)
    SetHistoScaling(LS1, kSigma)
    DS1 = DatasetModule.Dataset(LS1,
        Int(ds_num),
        joinpath(path_superreduced, "SuperReduced_Background_ds$(ds_num).root"),
        Float64(dict_2d[ds_num]["E_cut"]),
        Float64(dict_2d[ds_num]["err_cut"]),
        Float64(dict_2d[ds_num]["E_mc"]),
        Float64(dict_2d[ds_num]["err_mc"]),
        Float64(dict_2d[ds_num]["deltaT"]),
        Float64(dict_2d[ds_num]["scaling"]), #Q-value scaling
        Float64(dict_2d[ds_num]["err_scaling"]),
        1.0, #sigma scaling
        0.0,
        Float64(Emin),
        Float64(Emax),
        true,
        false,
        false,
        file_eff,
        "",
        joinpath(path_exposure, "Exposures_ds$(ds_num).txt"),
        "",
        true,
        file_lsscaling
    )
    push!(datasets, DS1)
    
end

dict_2d[3801]["E_cut"] = 0.9055
LineshapeGauss3 initialized for DS3819 with 882 active channels.


┌ Warning: For DsCh: 3819 -- 5  2615 sigma < baseline sigma: 
│   3.0132822557771948  < 3.1277805602496245, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 79  2615 sigma < baseline sigma: 
│   2.2139581598601867  < 2.2628519759069206, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 82  2615 sigma < baseline sigma: 
│   2.76228208825143  < 4.505629506628355, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 90  2615 sigma < baseline sigma: 
│   4.091736840910847  < 4.324466342027649, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 103  

File exists.
Reading histogram 'CombinedEff_DS3819' from /Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/CombinedEfficiency/CombinedEfficiencies.root ...


┌ Warning: For DsCh: 3819 -- 726  2615 sigma < baseline sigma: 
│   4.241626339943759  < 4.325595633372637, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 728  2615 sigma < baseline sigma: 
│   3.873954821909601  < 4.06173153257553, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 729  2615 sigma < baseline sigma: 
│   3.27769860278437  < 3.416766977414918, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 817  2615 sigma < baseline sigma: 
│   2.5508793514832298  < 2.6670017744281154, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3819 -- 843

LS scaling file exists.
Reading LS scaling histograms from /Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/lineshape_scaling_output/CombinedReso_model2_pol2bias.root ...
The histogram priors may be a bit odd out at the tails, so truncate them at percentile of 0.997
Reading histogram 'bias_Qbb_ds3819' ...
Reading histogram 'reso_Qbb_ds3819' ...
Opening file /Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/super_reduced/background/unblinded/SuperReduced_Background_ds3819.root
Number of entries in tree: 2490232


┌ Warning: No input ASCII for trigger efficiency by-channel. Setting all to 1.
└ @ Main.DatasetModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Dataset.jl:379
┌ Warning: You are calling a function to override the exposure with option to override disabled
└ @ Main.DatasetModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Dataset.jl:426
[ Info: Dataset initialized: DS=3819, Events=155, Exposure sum=94.72272199999999 kg·yr
┌ Warning: For DsCh: 3820 -- 7  2615 sigma < baseline sigma: 
│   3.5547954228119965  < 3.6197346487798407, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3820 -- 52  2615 sigma < baseline sigma: 
│   2.0459513529383577  < 2.1278769586524886, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3820 -- 82  2615 sigma < baseline sigma: 


LineshapeGauss3 initialized for DS3820 with 828 active channels.
File exists.
Reading histogram 'CombinedEff_DS3820' from /Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/CombinedEfficiency/CombinedEfficiencies.root ...


┌ Warning: For DsCh: 3820 -- 767  2615 sigma < baseline sigma: 
│   4.946070595347468  < 5.126387910374118, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3820 -- 780  2615 sigma < baseline sigma: 
│   3.085899547415532  < 3.2689141678720666, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3820 -- 815  2615 sigma < baseline sigma: 
│   2.6114601134605335  < 2.625283834422299, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3820 -- 817  2615 sigma < baseline sigma: 
│   3.022970581935409  < 3.1400256022807618, will set cal baseline = 2615 sigma
└ @ Main.LineshapeModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Lineshape.jl:245
┌ Warning: For DsCh: 3820 -- 

LS scaling file exists.
Reading LS scaling histograms from /Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/lineshape_scaling_output/CombinedReso_model2_pol2bias.root ...
The histogram priors may be a bit odd out at the tails, so truncate them at percentile of 0.997
Reading histogram 'bias_Qbb_ds3820' ...
Reading histogram 'reso_Qbb_ds3820' ...
Opening file /Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/super_reduced/background/unblinded/SuperReduced_Background_ds3820.root
Number of entries in tree: 1967948


┌ Warning: No input ASCII for trigger efficiency by-channel. Setting all to 1.
└ @ Main.DatasetModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Dataset.jl:379
┌ Warning: You are calling a function to override the exposure with option to override disabled
└ @ Main.DatasetModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Dataset.jl:426
[ Info: Dataset initialized: DS=3820, Events=107, Exposure sum=79.50047450000002 kg·yr


In [2]:
using OrderedCollections
datasets_group = [[datasets[1]], [datasets[2]]]
# datasets_group = [[datasets[1]]]
BI_priors = OrderedDict()
numm2::Int32 = 1
for dss in datasets_group
    BI_priors[Symbol("BI_$(numm2)")] = BI_prior(dss)
    global numm2 += 1
end
priors = OrderedDict()
priors[Symbol("Gamma_0ν")] = signal_prior(datasets)
merge!(priors, BI_priors)
priors[Symbol("Co60")] = Co60_prior(datasets)
merge!(priors,cutEff_prior(datasets))
priors[Symbol("Qββ")] = Qββ_prior()
prior_tmp = LineshapeScaling_prior(datasets, true)
for i in 1:1:length(prior_tmp)
    merge!(priors, prior_tmp[i])
end

Dataset 3819: 0 channels with zero efficiency or zero exposure in ROI.
Dataset 3820: 0 channels with zero efficiency or zero exposure in ROI.
Dataset 3819: 0 channels with zero efficiency or zero exposure in ROI.
Dataset 3820: 0 channels with zero efficiency or zero exposure in ROI.
Dataset 3819: 0 channels with zero efficiency or zero exposure in ROI.
Dataset 3820: 0 channels with zero efficiency or zero exposure in ROI.


┌ Warning: Lineshape scaling still under development. Run at your own risk.
└ @ Main.UtilsModule ~/gssiwork/Git_submit/Bayesian0nbbAnalysisCUOREJulia/src/Util.jl:391


Original bin range: start = '1', end = '92'.
Trimmed bin range: start = '1', end = '92'.
Original bin range: start = '1', end = '214'.
Trimmed bin range: start = '1', end = '214'.
Original bin range: start = '1', end = '107'.
Trimmed bin range: start = '1', end = '107'.
Original bin range: start = '1', end = '248'.
Trimmed bin range: start = '1', end = '248'.


In [5]:
using BAT
using Distributions
using Random
using StatsPlots
using ValueShapes
using LinearAlgebra
using Statistics
using DensityInterface

prior_my = distprod(; priors...)

NamedTupleDist((Gamma_0ν = Uniform{Float64}(a=0.0, b=7.357387838108791e-25), BI_1 = Uniform{Float64}(a=0.0, b=0.034845767426792344), BI_2 = Uniform{Float64}(a=0.0, b=0.030592363055475705), Co60 = Uniform{Float64}(a=0.0, b=2.072454578101862), EffCut_ds3819 = Main.HistoPriorModule.HistogramAsUvDistribution{Float64}(
h: StatsBase.Histogram{Float64, 1, Tuple{Vector{Float64}}}
edges:
  [0.921, 0.922, 0.923, 0.924, 0.925, 0.926, 0.927, 0.928, 0.929, 0.93  …  0.955, 0.956, 0.957, 0.958, 0.959, 0.96, 0.961, 0.962, 0.963, 0.964]
weights: [0.3303945075617538, 0.5206216482791272, 0.7696690232972674, 1.2514943468248252, 1.8096608255086972, 2.64565904918768, 3.598046247121372, 4.909612322593789, 6.44018990876055, 8.348718787668409  …  33.67520988436239, 28.36767241925218, 23.196497778172006, 17.684916674755474, 12.834124586462453, 8.69413122739206, 5.552880416861749, 3.367771287305604, 1.7470861081674558, 0.8835550088583266]
closed: left
isdensity: true
inv_weights: [3.0266846969696997, 1.920780673

In [ ]:
f_norm = PhysicalConstants.N_A * 1000. * PhysicalConstants.Abundance_130Te / PhysicalConstants.mass_TeO2
likelihood = DensityInterface.logfuncdensity(
    function (param::NamedTuple)
        total_ll = 0.0
        par_signal = param[:Gamma_0ν]
        par_co60 = param[Symbol("Co60")]
        par_Qbb = param[:Qββ]
        numm::Int32 = 1
        par_BI = []
        for datasets in datasets_group
            push!(par_BI, param[Symbol("BI_$(numm)")])
            numm += 1
            for ds in datasets
                par_bi = par_BI[numm-1]
                par_bias = param[Symbol("bias_ds$(ds.ds)")]*1.0
                par_reso = param[Symbol("reso_ds$(ds.ds)")]*1.0
                par_effcut = param[Symbol("EffCut_ds$(ds.ds)")]*1.0
                ls = ds.lineshape
                SetTmpPolyScaling(ls, kQvalue, [par_bias])
                SetTmpPolyScaling(ls, kSigma, [par_reso])
                for (ch, expo) in ds.exposure_channel
                    s = par_signal * f_norm * par_effcut * ds.containment_efficiency * ds.trigger_efficiency_channel[ch] * ds.exposure_channel[ch]
                    b = par_bi * (ds.emax - ds.emin) * par_effcut * ds.exposure_channel[ch]
                    c = par_co60 * par_effcut * exp(-(ds.delta_t) / PhysicalConstants.Tau60Cobalt) * ds.trigger_efficiency_channel[ch] * ds.exposure_channel[ch]
                    λ = s + b + c
                    total_ll -= λ
                    if !haskey(ds.events_channel, ch)
                        continue
                    else
                        events = ds.events_channel[ch]
                    end
                    for ev in events
                        energy = get_energy(ev)
                        total_ll += log(
                            s * GetResponsePDF(ls, ch, energy, par_Qbb*1.0)
                            + c * GetResponsePDF(ls, ch, energy, PhysicalConstants.ESumPeak60Cobalt)
                            + b/(ds.emax-ds.emin)
                        )
                    end
                end
            end
        end
        return total_ll
    end,
)
posterior = PosteriorDensity(likelihood, prior_my)

using CPUTime
algorithm = MCMCSampling(
    mcalg = MetropolisHastings(),
    nsteps = 10^5,
    nchains = 5,
    burnin = MCMCMultiCycleBurnin(nsteps_per_cycle = 10000, max_ncycles = 50),  # 简化burnin
)
CPUtic()
@time @CPUtime samples = bat_sample(posterior, algorithm)
cpu_time = CPUtoc()
println("Run completed in $cpu_time seconds")

In [6]:
# 可视化结果
using Plots
using BAT, HDF5

samples = bat_read("./program/result/SignalIncluded_Jan30_llup_HMC.h5")

# 请替换为您的第二个文件名
num_params = length(priors)
# 计算布局：每行最多3个图
ncols = min(3, num_params)
nrows = ceil(Int, num_params / ncols)

p = plot(size=(3*400*ncols, 3*300*nrows), 
         layout=(nrows, ncols), 
         labelfontsize=12, 
         tickfontsize=10, 
         legendfontsize=9)
num_h = 1
for key in keys(priors)
    row = ceil(Int, num_h / ncols)
    col = num_h - (row-1)*ncols
    plot!(p, samples.result, key, 
          subplot=num_h, 
          label="posterior", 
          linewidth=2,
          title=string(key),
          titlefontsize=11
    )
    pars = []
    idx = nothing
    for samp in samples.result
        v = samp.v
        weight = samp.weight
        for w = 1:1:weight
            if (idx == nothing)
                append!(pars, v[key])
            else
                append!(pars, v[key][idx])
            end
        end
    end
    x = range(minimum(pars), stop = maximum(pars), length = 100)
    y = pdf(prior_my[key], x)
    plot!(p, x, y,
              subplot=num_h,
              label="Prior",
              linewidth=3,
              color=:black,
              linestyle=:dash)
    num_h += 1
end
# 保存为单个PDF文件
savefig(p, "test_Feb2.pdf")

[ Info: Using input algorithm BATHDF5IO()
┌ Warning: No strict ticks found
└ @ PlotUtils ~/.julia/packages/PlotUtils/HX80C/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils ~/.julia/packages/PlotUtils/HX80C/src/ticks.jl:194


"/Users/zhaokangkang/gssiwork/julia-dev/code-test/test_Feb2.pdf"